In [ ]:
import json
from qdrant_client import QdrantClient
import time
from tqdm import tqdm
import os

# --- 配置参数 (请根据你的情况修改) ---

# 1. 连接到 Qdrant
# 方式一: 本地存储
QDRANT_PATH = "./exp_data/locomo1_failed/locomo1_failed_top_k_30_filter_False_graph_False_20250928_180635/qdrant_data"
client = QdrantClient(path=QDRANT_PATH)

# 2. 指定要导出的集合名称
COLLECTION_NAME = "mem0"  # 替换成你的集合名称

# 3. 指定输出的 JSON 文件路径
OUTPUT_JSON_FILE = os.path.join(QDRANT_PATH, "all_memory.json")


def export_qdrant_collection_to_json(client, collection_name, output_file):
    """
    从 Qdrant 集合中导出所有数据点并保存为 JSON 文件。
    """
    print(f"正在从集合 '{collection_name}' 中导出数据...")

    try:
        # 获取集合总数，用于进度条
        total_points = client.count(collection_name=collection_name, exact=True).count
        print(f"集合总点数: {total_points}")

        all_points = []
        next_page_offset = None

        with tqdm(total=total_points, desc="导出进度", unit="点") as pbar:
            while True:
                records, next_page_offset = client.scroll(
                    collection_name=collection_name,
                    limit=200,
                    offset=next_page_offset,
                    with_payload=True,
                    with_vectors=False
                )

                all_points.extend(records)
                pbar.update(len(records))  # 更新进度条
                if len(all_points) >= total_points or next_page_offset is None:
                    break

        print(f"\n数据拉取完成，总共 {len(all_points)} 个数据点。")

        # 转换为标准字典
        export_data = [
            {
                "id": point.id,
                "payload": point.payload
                # "vector": point.vector  # 如果需要向量，可以开启
            }
            for point in all_points
        ]

        print(f"正在将数据写入到 '{output_file}'...")
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(export_data, f, indent=4, ensure_ascii=False)

        print(f"\n✅ 导出成功！数据已保存到 {output_file}")

    except Exception as e:
        print(f"\n❌ 导出过程中发生错误: {e}")
        print("请检查以下几点：")
        print(f"1. Qdrant 实例是否正在运行或路径 '{QDRANT_PATH}' 是否正确。")
        print(f"2. 集合名称 '{collection_name}' 是否存在。")


if __name__ == "__main__":
    export_qdrant_collection_to_json(client, COLLECTION_NAME, OUTPUT_JSON_FILE)


正在从集合 'mem0' 中导出数据...
集合总点数: 412


导出进度: 100%|██████████| 412/412 [00:00<00:00, 208199.19点/s]


数据拉取完成，总共 412 个数据点。
正在将数据写入到 './exp_data/locomo1_failed/locomo1_failed_top_k_30_filter_False_graph_False_20250928_180635/qdrant_data/qdrant_export.json'...

✅ 导出成功！数据已保存到 ./exp_data/locomo1_failed/locomo1_failed_top_k_30_filter_False_graph_False_20250928_180635/qdrant_data/qdrant_export.json
